# Appendix: Models Used in This Course

Every part in this course calls some model or other -- a chat model, a tokenizer, an embedding model -- and their names (`"HuggingFaceTB/SmolLM2-135M"`, `"gemini-embedding-001"`...) tend to fly by without much said about what's actually behind them.

Rather than a hand-written table that goes stale the moment a new part changes a model name, this notebook **discovers the list itself**: it scans every `Part_*.ipynb` file in this repository for model identifiers, then looks each one up on the Hugging Face Hub for its real architecture facts (layers, attention heads, hidden size). Re-run it any time -- after adding a new part, or swapping a model -- and it reflects the course as it actually is today, not as it was when this cell was last edited by hand.

In [ ]:
# Program 1: scan every Part_*.ipynb in this repository for model identifiers

import glob
import json
import re

# Matches model = "...", model_name = "...", embed_name = "..." -- as a plain assignment
# or as a model="..." keyword argument, since both look identical as text.
MODEL_PATTERN = re.compile(r'''(?:model(?:_name)?|embed_name)\s*=\s*["']([^"']+)["']''')

def find_models(notebook_path):
    notebook = json.load(open(notebook_path))
    found = set()
    for cell in notebook["cells"]:
        if cell["cell_type"] != "code":
            continue
        source = "".join(cell["source"])
        found.update(MODEL_PATTERN.findall(source))
    return found

models_by_part = {}
for path in sorted(glob.glob("Part_*.ipynb")):
    for model_id in find_models(path):
        models_by_part.setdefault(model_id, set()).add(path)

for model_id, parts in sorted(models_by_part.items()):
    print(f"{model_id:35s} used in: {', '.join(sorted(parts))}")

## Looking up what each one actually is

For any model that lives on the Hugging Face Hub, `AutoConfig.from_pretrained` -- a lighter cousin of `AutoModel.from_pretrained`, since it only downloads the architecture description, not the weights -- gives real, current facts straight from the model's own config file.

For everything else -- Gemini's models, and anything served through the University of Rennes API or through Ollama under its own naming -- there's no equivalent lookup: closed APIs don't expose their architecture, and Ollama tags don't map cleanly onto a Hub repository. Those rows are marked **not exposed** rather than guessed at.

In [ ]:
# Program 2: for every model discoverable on the Hugging Face Hub, pull its real architecture facts

from IPython.display import Markdown, display
from transformers import AutoConfig

NOT_EXPOSED = "*not exposed*"

rows = []
for model_id in sorted(models_by_part):
    parts = ", ".join(p.replace("Part_", "").replace(".ipynb", "") for p in sorted(models_by_part[model_id]))
    try:
        config = AutoConfig.from_pretrained(model_id)
        rows.append((model_id, parts, config.model_type,
                     str(getattr(config, "hidden_size", NOT_EXPOSED)),
                     str(getattr(config, "num_hidden_layers", NOT_EXPOSED)),
                     str(getattr(config, "num_attention_heads", NOT_EXPOSED))))
    except Exception:
        rows.append((model_id, parts, "hosted API / non-Hub runtime", NOT_EXPOSED, NOT_EXPOSED, NOT_EXPOSED))

table = "| Model | Used in | Architecture | Hidden size | Layers | Attention heads |\n"
table += "|---|---|---|---|---|---|\n"
for model_id, parts, arch, hidden, layers, heads in rows:
    table += f"| `{model_id}` | {parts} | {arch} | {hidden} | {layers} | {heads} |\n"

display(Markdown(table))

## Reading this table

* **Hosted / non-Hub rows** aren't a failure of the lookup -- `ilaas/mistral-small-4-119b` and `ilaas/qwen-3.6-35b-instruct` (University of Rennes), the `gemini-*` models (Google), and `mistral:latest` (Ollama's own tag, not a Hub repository name) are all real models; their architecture just isn't published anywhere this notebook can query. Parameter counts and other size hints for these are only ever the ones stated in the course's own prose, where known.
* This is a plain regex scan over `model`/`model_name`/`embed_name` assignments, not a full Python parser -- it's accurate for how this course happens to name things, but sanity-check the list if a part ever renames its model variable to something unusual.
* Parameter counts ("135M", "118M"...) are deliberately left out of the auto-pulled columns: they're already stated in the relevant part's own prose, and computing them here would mean fully downloading each model's weights just to build a reference table.